# Solution R5 · Warm-up: KNN, the value of k and the effect of scale

**Type**: Implement with scikit-learn

## Objective

Train a KNN on 13 chemical measurements of wines with very different scales, compare `k = 1, 5, 15` **without and with** standardization and find which variable dominates the distance.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_wine(as_frame=True)
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(X_tr.shape, X_te.shape)

## Commented solution

In [ ]:
table = pd.DataFrame(index=[1, 5, 15], columns=["unscaled", "scaled"], dtype=float)
for k in table.index:
    plain = KNeighborsClassifier(k).fit(X_tr, y_tr)
    scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(k)).fit(X_tr, y_tr)
    table.loc[k] = [plain.score(X_te, y_te), scaled.score(X_te, y_te)]
print(table.round(3).to_string())
mean_gain = float((table["scaled"] - table["unscaled"]).mean())
print(f"mean gain of the standardization: {mean_gain:.3f}")

In [ ]:
std = X_tr.std()
dominant_feature = std.idxmax()
scale_ratio = float(std.max() / std.min())
print(std.round(2).sort_values().tail(4).to_string())
print(f"dominant: {dominant_feature} | ratio between the largest and the smallest scale: {scale_ratio:.0f}")

## Check

In [ ]:
with verify():
    check(table.notna().all().all() and (table["scaled"] > 0.93).all(), "with scaling the accuracy is above 0.93 for all the k", "make_pipeline(StandardScaler(), KNeighborsClassifier(k))")
    check((table["unscaled"] < 0.8).all() and (table["scaled"] - table["unscaled"] > 0.2).all(), "without scaling the accuracy stays below 0.8 and the standardization gains over 0.2 for each k", "compare the two columns")
    check(mean_gain > 0.25, "mean gain of the standardization over 0.25", "(table['scaled'] - table['unscaled']).mean()")
    check(dominant_feature == "proline" and scale_ratio > 1000, "proline dominates the distance and the ratio between the largest and the smallest scale exceeds 1000", "X_tr.std().idxmax()")
done()

## What you learned

- **Without scaling, a single variable decides:** `proline` has a standard deviation of 327, against 0.1-14 for the others (a ratio of 2685 between the largest and the smallest scale): in the Euclidean distance the difference in `proline` crushes all the others. The accuracy without scaling is 0.72, 0.72 and 0.67 (for `k = 1, 5, 15`), that is just above predicting a single cultivar.
- **With `StandardScaler` the accuracy rises to 0.96-1.0** for the same `k`: the mean gain is about 0.28. It is the difference between an unusable model and an almost perfect one, obtained with one line of code.
- **The scaler goes inside the `Pipeline`:** that way it is fitted only on the training set and the same treatment is applied to the test set (module 3). Comparing the `k` on a test set of 54 examples, as here, is only an illustration: to choose `k` you use cross-validation (lesson 5.3).

## Common pitfall

Applying a distance-based model to variables with different units without standardizing. No error, no warning: just a much worse model.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why does KNN need standardization and a tree does not?</summary>

KNN measures distances: a variable with a large scale dominates the computation. A tree uses thresholds on one variable at a time, so changing the units moves the thresholds but not the order of the examples and the result does not change.

</details>